# 証券営業インテリジェンス ハンズオン
## Part 0: 環境セットアップ

このノートブックでは Nomura Holdings 向け Cortex AI ハンズオンの環境を構築します。

### ハンズオン全体構成

| # | ノートブック | 内容 |
|---|---|---|
| 0 | **Part 0 (本書)** | 環境セットアップ・DDL |
| 1 | Part 1 | ダミーデータ生成 |
| 2 | Part 2 | Cortex AI Functions |
| 3 | Part 3 | Cortex Analyst（自然言語 to SQL）|
| 4 | Part 4 | Cortex Search（セマンティック検索）|
| 5 | Part 5 | Cortex Agent（Snowflake Intelligence）|

### デモシナリオ

> 山田太郎様（68歳・元上場企業役員・預かり資産8億円）から「孫の海外留学費用2,000万円のためにトヨタ株を売りたい」という相談が入りました。
> AIアシスタントを使って**最適な提案（証券担保ローン）**を10分で作成します。

### アーキテクチャ

```
Snowflake Intelligence (UI)
         │
   Cortex Agent（オーケストレーター）
   ├── Cortex Analyst（顧客資産DB）
   ├── Cortex Search（ニュース・書類・アナリストレポート）
   └── Cortex Analyst（信託商品DB）
```

In [ ]:
%%sql -r result_env
USE ROLE ACCOUNTADMIN;
CREATE DATABASE IF NOT EXISTS SNOWFINANCE_DB;
USE DATABASE SNOWFINANCE_DB;
CREATE SCHEMA IF NOT EXISTS DEMO_SCHEMA;
USE SCHEMA DEMO_SCHEMA;
CREATE WAREHOUSE IF NOT EXISTS DEMO_WH
    WAREHOUSE_SIZE = 'MEDIUM'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;
USE WAREHOUSE DEMO_WH;
-- Cortex AI のクロスリージョン推論を有効化
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';
SELECT 'Step 1: Environment ready!' AS STATUS;

-- Cortex Search / ファンド目論見書 用ウェアハウス（既存の場合はスキップ）
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH
    WAREHOUSE_SIZE = 'MEDIUM'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    COMMENT = 'Cortex Search インデックス構築用ウェアハウス';
SELECT 'COMPUTE_WH: OK' AS STATUS;

## Step 2: テーブル定義（DDL）

### データモデル概要

```
DIM_CUSTOMER (300人)
  ├── DIM_FAMILY (600件)          -- 家族構成・相続人情報
  ├── DIM_LIFE_EVENT (200件)      -- ライフイベント
  ├── FACT_PORTFOLIO (1,500件)    -- 保有資産
  └── FACT_TRANSACTION (500件)   -- 取引履歴

DIM_TRUST_PRODUCT (10商品)
  └── DIM_PRODUCT_RECOMMENDATION (30件)

NEWS_ARTICLES (50件)     → Cortex Search
LOAN_PRODUCT_DOCS (15件) → Cortex Search
ANALYST_REPORTS (30件)   → Cortex Search
```

In [ ]:
%%sql -r result_ddl1
-- 顧客マスタ（50名の富裕層顧客情報）
CREATE OR REPLACE TABLE DIM_CUSTOMER (
    CUSTOMER_ID               VARCHAR(20)   PRIMARY KEY,
    CUSTOMER_NAME             VARCHAR(100)  NOT NULL,
    CUSTOMER_NAME_KANA        VARCHAR(100),
    AGE                       INT,
    GENDER                    VARCHAR(10),
    BIRTH_DATE                DATE,
    OCCUPATION                VARCHAR(100),
    COMPANY_NAME              VARCHAR(200),
    POSITION                  VARCHAR(100),
    PREFECTURE                VARCHAR(50),
    CITY                      VARCHAR(100),
    ANNUAL_INCOME_BAND        VARCHAR(50),
    TOTAL_ASSETS              DECIMAL(18,0),
    LIQUID_ASSETS             DECIMAL(18,0),
    RISK_TOLERANCE            VARCHAR(30),
    INVESTMENT_PURPOSE        VARCHAR(50),
    INVESTMENT_EXPERIENCE_YEARS INT,
    SEGMENT                   VARCHAR(50),
    RM_ID                     VARCHAR(20),
    RM_NAME                   VARCHAR(100),
    ACCOUNT_OPEN_DATE         DATE,
    LAST_CONTACT_DATE         DATE,
    HAS_NISA                  BOOLEAN DEFAULT FALSE,
    HAS_IDECO                 BOOLEAN DEFAULT FALSE,
    HAS_TRUST_ACCOUNT         BOOLEAN DEFAULT FALSE,
    NOTES                     TEXT,
    CREATED_AT                TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    UPDATED_AT                TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);
COMMENT ON TABLE DIM_CUSTOMER IS '顧客マスタ。富裕層の属性・担当RM情報';

-- 家族構成（相続人フラグ付き）
CREATE OR REPLACE TABLE DIM_FAMILY (
    FAMILY_ID         VARCHAR(20) PRIMARY KEY,
    CUSTOMER_ID       VARCHAR(20) NOT NULL REFERENCES DIM_CUSTOMER,
    RELATIONSHIP      VARCHAR(30) NOT NULL,
    FAMILY_NAME       VARCHAR(100),
    FAMILY_AGE        INT,
    FAMILY_OCCUPATION VARCHAR(100),
    IS_HEIR           BOOLEAN DEFAULT FALSE,
    HEIR_PRIORITY     INT,
    NOTES             TEXT,
    CREATED_AT        TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);
COMMENT ON TABLE DIM_FAMILY IS '顧客の家族構成。相続対策の提案に使用';

-- ライフイベント（教育・相続・不動産購入等）
CREATE OR REPLACE TABLE DIM_LIFE_EVENT (
    EVENT_ID          VARCHAR(20) PRIMARY KEY,
    CUSTOMER_ID       VARCHAR(20) NOT NULL REFERENCES DIM_CUSTOMER,
    EVENT_TYPE        VARCHAR(50) NOT NULL,
    EVENT_DETAIL      TEXT,
    EXPECTED_DATE     DATE,
    ESTIMATED_AMOUNT  DECIMAL(18,0),
    URGENCY           VARCHAR(20),
    STATUS            VARCHAR(30),
    RELATED_FAMILY_ID VARCHAR(20),
    CREATED_DATE      DATE,
    UPDATED_DATE      DATE,
    NOTES             TEXT
);
COMMENT ON TABLE DIM_LIFE_EVENT IS 'ライフイベント（教育・相続・不動産等）';

SELECT 'Customer dim tables: OK' AS STATUS;

In [ ]:
%%sql -r result_ddl2
-- ポートフォリオ（保有資産明細）
CREATE OR REPLACE TABLE FACT_PORTFOLIO (
    PORTFOLIO_ID        VARCHAR(20)   PRIMARY KEY,
    CUSTOMER_ID         VARCHAR(20)   NOT NULL REFERENCES DIM_CUSTOMER,
    ASSET_CLASS         VARCHAR(50)   NOT NULL,
    SECURITY_CODE       VARCHAR(20),
    SECURITY_NAME       VARCHAR(200)  NOT NULL,
    QUANTITY            DECIMAL(18,4),
    ACQUISITION_PRICE   DECIMAL(18,4),
    ACQUISITION_DATE    DATE,
    CURRENT_PRICE       DECIMAL(18,4),
    MARKET_VALUE        DECIMAL(18,0),
    UNREALIZED_GAIN     DECIMAL(18,0),
    UNREALIZED_GAIN_PCT DECIMAL(10,2),
    ACCOUNT_TYPE        VARCHAR(30),
    CURRENCY            VARCHAR(10)   DEFAULT 'JPY',
    AS_OF_DATE          DATE
);
COMMENT ON TABLE FACT_PORTFOLIO IS '保有資産明細（含み益・含み損含む）';

-- 取引履歴
CREATE OR REPLACE TABLE FACT_TRANSACTION (
    TRANSACTION_ID   VARCHAR(20) PRIMARY KEY,
    CUSTOMER_ID      VARCHAR(20) NOT NULL REFERENCES DIM_CUSTOMER,
    TRANSACTION_DATE DATE        NOT NULL,
    SETTLEMENT_DATE  DATE,
    TRANSACTION_TYPE VARCHAR(20) NOT NULL,
    ASSET_CLASS      VARCHAR(50),
    SECURITY_CODE    VARCHAR(20),
    SECURITY_NAME    VARCHAR(200),
    QUANTITY         DECIMAL(18,4),
    PRICE            DECIMAL(18,4),
    AMOUNT           DECIMAL(18,0),
    FEE              DECIMAL(18,0),
    TAX              DECIMAL(18,0),
    NET_AMOUNT       DECIMAL(18,0),
    ACCOUNT_TYPE     VARCHAR(30),
    ORDER_CHANNEL    VARCHAR(30),
    RM_ID            VARCHAR(20),
    NOTES            TEXT
);
COMMENT ON TABLE FACT_TRANSACTION IS '取引履歴（売買・入出金）';

SELECT 'Fact tables: OK' AS STATUS;

In [ ]:
%%sql -r result_ddl3
-- 信託銀行商品（証券担保ローン・教育資金贈与信託等）
CREATE OR REPLACE TABLE DIM_TRUST_PRODUCT (
    PRODUCT_ID          VARCHAR(20)  PRIMARY KEY,
    PRODUCT_NAME        VARCHAR(200) NOT NULL,
    PRODUCT_NAME_EN     VARCHAR(200),
    PRODUCT_CATEGORY    VARCHAR(50)  NOT NULL,
    PRODUCT_SUBCATEGORY VARCHAR(50),
    DESCRIPTION         TEXT,
    MIN_AMOUNT          DECIMAL(18,0),
    MAX_AMOUNT          DECIMAL(18,0),
    INTEREST_RATE_MIN   DECIMAL(5,2),
    INTEREST_RATE_MAX   DECIMAL(5,2),
    TERM_MIN_MONTHS     INT,
    TERM_MAX_MONTHS     INT,
    ELIGIBLE_SEGMENT    VARCHAR(100),
    ELIGIBLE_ASSETS_MIN DECIMAL(18,0),
    TAX_BENEFIT         TEXT,
    RISKS               TEXT,
    IS_ACTIVE           BOOLEAN DEFAULT TRUE,
    LAUNCH_DATE         DATE,
    PROVIDER            VARCHAR(100) DEFAULT 'プレミアム信託銀行'
);
COMMENT ON TABLE DIM_TRUST_PRODUCT IS '信託銀行の商品マスタ（証券担保ローン・教育信託等）';

-- 商品推奨ロジック（どの顧客にどの商品を提案すべきか）
CREATE OR REPLACE TABLE DIM_PRODUCT_RECOMMENDATION (
    RECOMMENDATION_ID           VARCHAR(20) PRIMARY KEY,
    PRODUCT_ID                  VARCHAR(20) NOT NULL REFERENCES DIM_TRUST_PRODUCT,
    TARGET_LIFE_EVENT           VARCHAR(50),
    TARGET_AGE_MIN              INT,
    TARGET_AGE_MAX              INT,
    TARGET_ASSETS_MIN           DECIMAL(18,0),
    TARGET_ASSETS_MAX           DECIMAL(18,0),
    TARGET_RISK_TOLERANCE       VARCHAR(30),
    TARGET_SEGMENT              VARCHAR(50),
    RECOMMENDATION_REASON       TEXT,
    BENEFIT_DESCRIPTION         TEXT,
    COMPARISON_WITH_ALTERNATIVE TEXT,
    PRIORITY                    INT DEFAULT 5,
    IS_ACTIVE                   BOOLEAN DEFAULT TRUE
);
COMMENT ON TABLE DIM_PRODUCT_RECOMMENDATION IS '商品推奨ロジック';

-- Cortex Search 用: マーケットニュース（50件）
CREATE OR REPLACE TABLE NEWS_ARTICLES (
    NEWS_ID            VARCHAR(20)  PRIMARY KEY,
    PUBLISH_DATE       DATE         NOT NULL,
    PUBLISH_DATETIME   TIMESTAMP,
    SOURCE             VARCHAR(100),
    CATEGORY           VARCHAR(50),
    TITLE              VARCHAR(500) NOT NULL,
    CONTENT            TEXT         NOT NULL,
    SUMMARY            TEXT,
    RELATED_SECURITIES VARCHAR(500),
    SENTIMENT          VARCHAR(20),
    IMPORTANCE         INT,
    TAGS               VARCHAR(500)
);
COMMENT ON TABLE NEWS_ARTICLES IS 'マーケットニュース・税制改正ニュース（50件）';

-- Cortex Search 用: ローン商品説明書（チャンク分割済み, 15件）
CREATE OR REPLACE TABLE LOAN_PRODUCT_DOCS (
    DOC_ID      VARCHAR(20) PRIMARY KEY,
    PRODUCT_ID  VARCHAR(20),
    DOC_TYPE    VARCHAR(50),
    SECTION     VARCHAR(100),
    TITLE       VARCHAR(200),
    CONTENT     TEXT         NOT NULL,
    PAGE_NUMBER INT,
    CHUNK_INDEX INT,
    CREATED_AT  TIMESTAMP    DEFAULT CURRENT_TIMESTAMP()
);
COMMENT ON TABLE LOAN_PRODUCT_DOCS IS 'ローン商品説明書（チャンク分割済み）';

-- Cortex Search 用: アナリストレポート（30件）
CREATE OR REPLACE TABLE ANALYST_REPORTS (
    REPORT_ID             VARCHAR(20)  PRIMARY KEY,
    PUBLISH_DATE          DATE         NOT NULL,
    SECURITY_CODE         VARCHAR(20),
    SECURITY_NAME         VARCHAR(200),
    ANALYST_NAME          VARCHAR(100),
    ANALYST_TEAM          VARCHAR(100),
    RATING                VARCHAR(20),
    PREVIOUS_RATING       VARCHAR(20),
    TARGET_PRICE          DECIMAL(18,0),
    PREVIOUS_TARGET_PRICE DECIMAL(18,0),
    CURRENT_PRICE         DECIMAL(18,0),
    UPSIDE_PCT            DECIMAL(10,2),
    REPORT_TITLE          VARCHAR(500),
    EXECUTIVE_SUMMARY     TEXT,
    INVESTMENT_THESIS     TEXT,
    KEY_RISKS             TEXT,
    EARNINGS_FORECAST     TEXT,
    CONTENT               TEXT
);
COMMENT ON TABLE ANALYST_REPORTS IS '証券アナリストレポート（30件）';

SELECT 'Trust & Search tables: OK' AS STATUS;

## Step 3: 分析用ビューの作成

**V_INHERITANCE_TAX_ESTIMATE**: 相続税簡易試算ビュー
- 基礎控除 = 3,000万円 + 600万円 × 法定相続人数（日本の相続税法）
- 累進税率 10%〜55%を適用

**V_PORTFOLIO_SUMMARY**: ポートフォリオ資産クラス別サマリービュー

In [ ]:
%%sql -r result_views
-- 相続税試算ビュー
CREATE OR REPLACE VIEW V_INHERITANCE_TAX_ESTIMATE AS
WITH heirs AS (
    SELECT CUSTOMER_ID, COUNT(*) AS NUM_HEIRS
    FROM DIM_FAMILY WHERE IS_HEIR = TRUE GROUP BY CUSTOMER_ID
)
SELECT
    c.CUSTOMER_ID, c.CUSTOMER_NAME, c.AGE, c.TOTAL_ASSETS,
    COALESCE(h.NUM_HEIRS, 1) AS NUM_HEIRS,
    30000000 + 6000000 * COALESCE(h.NUM_HEIRS, 1) AS BASIC_DEDUCTION,
    GREATEST(c.TOTAL_ASSETS - 30000000 - 6000000 * COALESCE(h.NUM_HEIRS, 1), 0) AS TAXABLE_ESTATE,
    CASE
        WHEN c.TOTAL_ASSETS - 30000000 - 6000000*COALESCE(h.NUM_HEIRS,1) <= 0
            THEN 0
        WHEN c.TOTAL_ASSETS - 30000000 - 6000000*COALESCE(h.NUM_HEIRS,1) <= 10000000
            THEN (c.TOTAL_ASSETS-30000000-6000000*COALESCE(h.NUM_HEIRS,1))*0.10
        WHEN c.TOTAL_ASSETS - 30000000 - 6000000*COALESCE(h.NUM_HEIRS,1) <= 30000000
            THEN 1000000+((c.TOTAL_ASSETS-30000000-6000000*COALESCE(h.NUM_HEIRS,1)-10000000)*0.15)
        WHEN c.TOTAL_ASSETS - 30000000 - 6000000*COALESCE(h.NUM_HEIRS,1) <= 50000000
            THEN 4000000+((c.TOTAL_ASSETS-30000000-6000000*COALESCE(h.NUM_HEIRS,1)-30000000)*0.20)
        WHEN c.TOTAL_ASSETS - 30000000 - 6000000*COALESCE(h.NUM_HEIRS,1) <= 100000000
            THEN 8000000+((c.TOTAL_ASSETS-30000000-6000000*COALESCE(h.NUM_HEIRS,1)-50000000)*0.30)
        WHEN c.TOTAL_ASSETS - 30000000 - 6000000*COALESCE(h.NUM_HEIRS,1) <= 200000000
            THEN 23000000+((c.TOTAL_ASSETS-30000000-6000000*COALESCE(h.NUM_HEIRS,1)-100000000)*0.40)
        WHEN c.TOTAL_ASSETS - 30000000 - 6000000*COALESCE(h.NUM_HEIRS,1) <= 300000000
            THEN 63000000+((c.TOTAL_ASSETS-30000000-6000000*COALESCE(h.NUM_HEIRS,1)-200000000)*0.45)
        WHEN c.TOTAL_ASSETS - 30000000 - 6000000*COALESCE(h.NUM_HEIRS,1) <= 600000000
            THEN 108000000+((c.TOTAL_ASSETS-30000000-6000000*COALESCE(h.NUM_HEIRS,1)-300000000)*0.50)
        ELSE 258000000+((c.TOTAL_ASSETS-30000000-6000000*COALESCE(h.NUM_HEIRS,1)-600000000)*0.55)
    END AS ESTIMATED_TAX,
    ROUND(GREATEST(c.TOTAL_ASSETS-30000000-6000000*COALESCE(h.NUM_HEIRS,1),0)*0.30
          /NULLIF(c.TOTAL_ASSETS,0)*100, 2) AS EFFECTIVE_TAX_RATE_PCT
FROM DIM_CUSTOMER c
LEFT JOIN heirs h ON c.CUSTOMER_ID = h.CUSTOMER_ID
WHERE c.TOTAL_ASSETS > 0;
COMMENT ON VIEW V_INHERITANCE_TAX_ESTIMATE IS '相続税概算ビュー（簡易計算）';

-- ポートフォリオサマリービュー
CREATE OR REPLACE VIEW V_PORTFOLIO_SUMMARY AS
SELECT
    c.CUSTOMER_ID, c.CUSTOMER_NAME, c.SEGMENT, c.TOTAL_ASSETS,
    p.ASSET_CLASS,
    SUM(p.MARKET_VALUE)    AS ASSET_CLASS_VALUE,
    ROUND(SUM(p.MARKET_VALUE)/NULLIF(c.TOTAL_ASSETS,0)*100, 2) AS ALLOCATION_PCT,
    SUM(p.UNREALIZED_GAIN) AS TOTAL_UNREALIZED_GAIN,
    ROUND(SUM(p.UNREALIZED_GAIN)/NULLIF(SUM(p.MARKET_VALUE-p.UNREALIZED_GAIN),0)*100,2) AS RETURN_PCT
FROM DIM_CUSTOMER c
JOIN FACT_PORTFOLIO p ON c.CUSTOMER_ID = p.CUSTOMER_ID
GROUP BY c.CUSTOMER_ID, c.CUSTOMER_NAME, c.SEGMENT, c.TOTAL_ASSETS, p.ASSET_CLASS;
COMMENT ON VIEW V_PORTFOLIO_SUMMARY IS 'ポートフォリオ資産クラス別サマリー';

SELECT 'Views: OK' AS STATUS;

In [ ]:
%%sql -r result_verify
SELECT TABLE_TYPE, TABLE_NAME, COMMENT
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'DEMO_SCHEMA'
ORDER BY TABLE_TYPE DESC, TABLE_NAME;

## まとめ

Part 0 完了！

| 作成したオブジェクト | 詳細 |
|---|---|
| Database | SNOWFINANCE_DB |
| Schema | DEMO_SCHEMA |
| Warehouse | DEMO_WH (MEDIUM) |
| Tables | 9テーブル |
| Views | 2ビュー |

**次のステップ:** `part1_data_generation.ipynb` を実行してダミーデータを投入します。